In [0]:
# ========================================
# CONFIG ODOO PRODUCCIÓN
# ========================================

BASE_URL = dbutils.secrets.get(
    "odoo-production",
    "base_url"
)

DB = dbutils.secrets.get(
    "odoo-production",
    "db"
)

UID = int(
    dbutils.secrets.get(
        "odoo-production",
        "uid"
    )
)

API_KEY = dbutils.secrets.get(
    "odoo-production",
    "api_key"
)

In [0]:
import requests

def odoo_call(
    model,
    method,
    args=None,
    kwargs=None
):

    if args is None:
        args = []

    if kwargs is None:
        kwargs = {}

    payload = {
        "jsonrpc": "2.0",
        "method": "call",
        "params": {
            "service": "object",
            "method": "execute_kw",
            "args": [
                DB,
                UID,
                API_KEY,
                model,
                method,
                args,
                kwargs
            ]
        }
    }

    response = requests.post(
        BASE_URL,
        json=payload,
        headers={
            "Content-Type": "application/json"
        },
        timeout=120
    )

    result = response.json()

    if "error" in result:
        raise Exception(result["error"])

    return result["result"]

In [0]:
users = odoo_call(
    model="res.users",
    method="search_read",
    args=[
        [
            ["active","=",True]
        ]
    ],
    kwargs={
        "fields":[
            "id",
            "name",
            "login"
        ],
        "limit":10
    }
)

display(users)

In [0]:
import requests

SCOPE = "odoo-production"

BASE_URL = dbutils.secrets.get(SCOPE, "base_url")
DB = dbutils.secrets.get(SCOPE, "db")
UID = int(dbutils.secrets.get(SCOPE, "uid"))
API_KEY = dbutils.secrets.get(SCOPE, "api_key")


def odoo_execute_kw(model, method, args=None, kwargs=None):
    args = args or []
    kwargs = kwargs or {}

    payload = {
        "jsonrpc": "2.0",
        "method": "call",
        "params": {
            "service": "object",
            "method": "execute_kw",
            "args": [
                DB,
                UID,
                API_KEY,
                model,
                method,
                args,
                kwargs
            ]
        }
    }

    response = requests.post(
        BASE_URL,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=180
    )

    response.raise_for_status()
    body = response.json()

    if "error" in body:
        error_data = body["error"].get("data", {})
        raise RuntimeError(
            error_data.get("message")
            or body["error"].get("message")
            or str(body["error"])
        )

    return body.get("result")

In [0]:
helpdesk_fields = odoo_execute_kw(
    model="ir.model.fields",
    method="search_read",
    args=[
        [
            ["model", "=", "helpdesk.ticket"]
        ]
    ],
    kwargs={
        "fields": [
            "name",
            "field_description",
            "ttype",
            "relation",
            "store",
            "readonly"
        ],
        "limit": 5000,
        "order": "name"
    }
)

display(helpdesk_fields)

In [0]:
terminos = [
    "sla",
    "response",
    "respuesta",
    "close",
    "cierre",
    "hours",
    "horas",
    "ticket",
    "team",
    "project"
]

campos_interes = [
    f for f in helpdesk_fields
    if any(
        t in (
            str(f.get("name", "")) + " " +
            str(f.get("field_description", ""))
        ).lower()
        for t in terminos
    )
]

display(campos_interes)


In [0]:
def odoo_search_read_all(
    model,
    fields,
    domain=None,
    batch_size=500,
    order="id"
):
    domain = domain or []
    records = []
    offset = 0

    while True:
        batch = odoo_execute_kw(
            model=model,
            method="search_read",
            args=[domain],
            kwargs={
                "fields": fields,
                "limit": batch_size,
                "offset": offset,
                "order": order
            }
        )

        if not batch:
            break

        records.extend(batch)

        print(
            f"{model}: offset={offset}, "
            f"lote={len(batch)}, acumulado={len(records)}"
        )

        if len(batch) < batch_size:
            break

        offset += batch_size

    return records


In [0]:
HELPDESK_FIELDS = [
    "id",
    "name",
    "priority",

    "partner_id",
    "partner_email",

    "user_id",
    "team_id",
    "stage_id",
    "ticket_type_id",

    "create_date",
    "write_date",

    "total_hours_spent",
    "tiempo_primera_respuesta"
]

In [0]:
CANDIDATE_FIELDS = [
    "ticket_ref",
    "close_date",
    "date_closed",
    "closed_date",
    "project_id",
    "sla_id",
    "sla_deadline",
    "sla_reached",
    "sla_status",
    "kanban_state",
    "description",
    "tag_ids"
]

In [0]:
available_fields = {f["name"] for f in helpdesk_fields}

HELPDESK_FIELDS_FINAL = [
    field
    for field in HELPDESK_FIELDS + CANDIDATE_FIELDS
    if field in available_fields
]

print(HELPDESK_FIELDS_FINAL)

In [0]:
tickets = odoo_search_read_all(
    model="helpdesk.ticket",
    fields=HELPDESK_FIELDS_FINAL,
    domain=[],
    batch_size=500,
    order="id"
)

print("Total tickets extraídos:", len(tickets))


In [0]:
from datetime import datetime, timedelta, timezone

watermark = (
    datetime.now(timezone.utc) - timedelta(days=2)
).strftime("%Y-%m-%d %H:%M:%S")

tickets_incrementales = odoo_search_read_all(
    model="helpdesk.ticket",
    fields=HELPDESK_FIELDS_FINAL,
    domain=[
        ["write_date", ">=", watermark]
    ],
    batch_size=500,
    order="write_date,id"
)

print("Tickets incrementales:", len(tickets_incrementales))

In [0]:

HELPDESK_FIELDS = [
    # Identificación
    "id",
    "ticket_ref",
    "name",
    "active",
    "description",

    # Clasificación
    "priority",
    "ticket_origin",
    "ticket_type_id",
    "line_type_id",
    "tag_ids",
    "kanban_state",

    # Organización
    "user_id",
    "team_id",
    "stage_id",
    "company_id",

    # Cliente y entidad
    "partner_id",
    "commercial_partner_id",
    "partner_name",
    "partner_email",
    "partner_phone",

    # Fechas
    "create_date",
    "write_date",
    "assign_date",
    "close_date",
    "date_last_stage_update",

    # Tiempos operativos
    "assign_hours",
    "close_hours",
    "open_hours",
    "first_response_hours",
    "avg_response_hours",
    "total_response_hours",
    "total_hours_spent",
    "count_first_response_timer",
    "count_resolution_timer",

    # Campos SLA custom
    "x_studio_criticidad",
    "x_studio_sla",
    "tiempo_primera_respuesta",
    "tiempo_primera_respuesta_esperado",
    "tiempo_resolucion_esperado",
    "tiempo_primera_respuesta_formatted",
    "total_hours_spent_formatted",

    # SLA nativo Odoo
    "use_sla",
    "sla_ids",
    "sla_status_ids",
    "sla_deadline",
    "sla_deadline_hours",
    "sla_reached",
    "sla_reached_late",
    "sla_fail",
    "sla_success",

    # Proyectos y venta
    "project_id",
    "project_sale_order_id",
    "sale_order_id",
    "sale_line_id",
    "analytic_account_id",

    # Horas y comunicación
    "timesheet_ids",
    "message_ids",
    "activity_ids",

    # Satisfacción
    "rating_last_value",
    "rating_last_feedback"
]

Filtra automáticamente contra la metadata

In [0]:
available_fields = {f["name"] for f in helpdesk_fields}

HELPDESK_FIELDS_FINAL = [
    field
    for field in HELPDESK_FIELDS
    if field in available_fields
]

missing_fields = sorted(
    set(HELPDESK_FIELDS) - available_fields
)

print("Campos disponibles:", len(HELPDESK_FIELDS_FINAL))
print(HELPDESK_FIELDS_FINAL)

print("\nCampos no encontrados:", len(missing_fields))
print(missing_fields)

Ejecutar la extracción completa

In [0]:
tickets_full = odoo_search_read_all(
    model="helpdesk.ticket",
    fields=HELPDESK_FIELDS_FINAL,
    domain=[],
    batch_size=500,
    order="id"
)

print("Total tickets extraídos:", len(tickets_full))


In [0]:
ticket_ids = [int(r["id"]) for r in tickets_full]

print("Registros extraídos:", len(ticket_ids))
print("Tickets únicos:", len(set(ticket_ids)))
print("Duplicados:", len(ticket_ids) - len(set(ticket_ids)))

In [0]:
datetime.now(timezone.utc) - timedelta(days=2)

In [0]:
%sql DESCRIBE CATALOG EXTENDED dailytech;

In [0]:
spark.sql("SHOW SCHEMAS IN dailytech")

In [0]:
%sql SHOW TABLES IN dailytech.gold

In [0]:
%sql
CREATE CATALOG corp_dailytech;
CREATE CATALOG data_platform;
CREATE CATALOG lakehouse_dailytech;

In [0]:
%sql


In [0]:
 %sql
 CREATE STORAGE CREDENTIAL sc_dailytech_prod;

In [0]:
%sql
CREATE TABLE corp_dailytech.bronze.raw_partners (
    source_model STRING,
    record_id BIGINT,
    payload STRING,
    source_write_date STRING,
    ingested_at TIMESTAMP,
    run_id STRING
)
USING DELTA;

In [0]:
%sql
CREATE TABLE corp_dailytech.bronze.raw_employees (
    source_model STRING,
    record_id BIGINT,
    payload STRING,
    source_write_date STRING,
    ingested_at TIMESTAMP,
    run_id STRING
)
USING DELTA;


In [0]:
%sql
SHOW TABLES IN corp_dailytech.config;


In [0]:
%sql
SELECT
    customer_name,
    COUNT(*) AS tickets
FROM corp_dailytech.silver.helpdesk_ticket
GROUP BY customer_name
ORDER BY tickets DESC;

In [0]:
%sql
SELECT
    commercial_partner_name,
    COUNT(*) AS tickets
FROM corp_dailytech.silver.helpdesk_ticket
GROUP BY commercial_partner_name
ORDER BY tickets DESC;